In [21]:
import sqlite3
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import StandardScaler

from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

DB_PATH = "../data/soccer.db"

ROLLING_WINDOW = 5
DRAW_ROLLING_WINDOW = 10
INITIAL_ELO = 1500
ELO_K = 20

EUROPEAN_LEAGUES = [1, 2, 3, 4, 5]

COMPETITION_ID_TO_NAME = {
    1: "Premier League",
    2: "La Liga",
    3: "Serie A",
    4: "Bundesliga",
    5: "Ligue 1",
}

FORECAST_COMPETITION_ID = 3
FORECAST_SEASON = 2025

In [15]:
conn = sqlite3.connect(DB_PATH)

matches_query = """
SELECT
    *
FROM matches
"""
matches_df = pd.read_sql(matches_query, conn)

context_query = """
SELECT
    match_id,
    home_position,
    away_position,
    points_diff,
    position_diff,
    home_title_race,
    away_title_race,
    home_europe_race,
    away_europe_race,
    home_relegation_risk,
    away_relegation_risk
FROM match_context
"""
context_df = pd.read_sql(context_query, conn)

matches_df["date"] = pd.to_datetime(matches_df["date"], errors="coerce")
context_df = context_df.drop(columns=["id"], errors="ignore")

for col in ["competition_id", "season", "home_team_id", "away_team_id", "home_goals", "away_goals"]:
    if col in matches_df.columns:
        matches_df[col] = pd.to_numeric(matches_df[col], errors="coerce")

matches_all = matches_df.merge(
    context_df,
    left_on="id",
    right_on="match_id",
    how="left"
)

matches_all = matches_all[
    matches_all["competition_id"].isin(EUROPEAN_LEAGUES)
].copy()

matches_all["competition_name"] = matches_all["competition_id"].map(COMPETITION_ID_TO_NAME)

completed_matches = matches_all[
    matches_all["status"].isin(["FT", "AET", "PEN"])
].copy()

completed_matches = completed_matches.sort_values(
    by=["date", "id"]
).reset_index(drop=True)

completed_matches["home_win"] = (
    completed_matches["home_goals"] > completed_matches["away_goals"]
)
completed_matches["away_win"] = (
    completed_matches["away_goals"] > completed_matches["home_goals"]
)
completed_matches["is_draw"] = (
    completed_matches["home_goals"] == completed_matches["away_goals"]
)
completed_matches["home_dnb"] = (
    completed_matches["home_goals"] >= completed_matches["away_goals"]
)
completed_matches["away_dnb"] = (
    completed_matches["away_goals"] >= completed_matches["home_goals"]
)
completed_matches["match_result_3class"] = np.select(
    [
        completed_matches["home_win"],
        completed_matches["is_draw"],
        completed_matches["away_win"],
    ],
    [0, 1, 2],
    default=np.nan,
)

In [16]:
home_history = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "home_team_id",
        "home_goals",
        "away_goals",
        "is_draw",
    ]
].copy()

home_history.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded",
    "draw_flag",
]

away_history = completed_matches[
    [
        "date",
        "competition_name",
        "season",
        "away_team_id",
        "away_goals",
        "home_goals",
        "is_draw",
    ]
].copy()

away_history.columns = [
    "date",
    "competition_name",
    "season",
    "team_id",
    "goals_scored",
    "goals_conceded",
    "draw_flag",
]

team_history = pd.concat([home_history, away_history], ignore_index=True)
team_history = team_history.sort_values(by=["team_id", "date"]).copy()

team_history["rolling_goals_scored"] = (
    team_history.groupby("team_id")["goals_scored"]
    .transform(lambda x: x.shift(1).rolling(ROLLING_WINDOW, min_periods=1).mean())
)

team_history["rolling_goals_conceded"] = (
    team_history.groupby("team_id")["goals_conceded"]
    .transform(lambda x: x.shift(1).rolling(ROLLING_WINDOW, min_periods=1).mean())
)

team_history["draw_rate_last_10"] = (
    team_history.groupby("team_id")["draw_flag"]
    .transform(lambda x: x.shift(1).rolling(DRAW_ROLLING_WINDOW, min_periods=1).mean())
)

home_features = team_history.rename(
    columns={
        "team_id": "home_team_id",
        "rolling_goals_scored": "home_rolling_scored",
        "rolling_goals_conceded": "home_rolling_conceded",
        "draw_rate_last_10": "home_draw_rate_last_10",
    }
)

away_features = team_history.rename(
    columns={
        "team_id": "away_team_id",
        "rolling_goals_scored": "away_rolling_scored",
        "rolling_goals_conceded": "away_rolling_conceded",
        "draw_rate_last_10": "away_draw_rate_last_10",
    }
)

elo_matches = completed_matches.sort_values(by=["date", "id"]).copy()
elo_ratings = {}
home_elo_values = []
away_elo_values = []

for _, match in elo_matches.iterrows():
    home_team = match["home_team_id"]
    away_team = match["away_team_id"]

    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    home_elo_values.append(home_elo)
    away_elo_values.append(away_elo)

    expected_home = 1 / (1 + 10 ** ((away_elo - home_elo) / 400))
    expected_away = 1 - expected_home

    if match["home_goals"] > match["away_goals"]:
        actual_home, actual_away = 1, 0
    elif match["home_goals"] < match["away_goals"]:
        actual_home, actual_away = 0, 1
    else:
        actual_home, actual_away = 0.5, 0.5

    elo_ratings[home_team] = home_elo + ELO_K * (actual_home - expected_home)
    elo_ratings[away_team] = away_elo + ELO_K * (actual_away - expected_away)

elo_history = pd.DataFrame({
    "id": elo_matches["id"].values,
    "home_elo": home_elo_values,
    "away_elo": away_elo_values,
})

team_snapshot = (
    team_history.sort_values(["team_id", "date"])
    .groupby("team_id", as_index=False)
    .tail(1)
    [["team_id", "rolling_goals_scored", "rolling_goals_conceded", "draw_rate_last_10"]]
    .copy()
)

elo_snapshot = pd.DataFrame(
    {
        "team_id": list(elo_ratings.keys()),
        "elo": list(elo_ratings.values()),
    }
)

team_snapshot = team_snapshot.merge(
    elo_snapshot,
    on="team_id",
    how="left"
)

team_snapshot["elo"] = team_snapshot["elo"].fillna(INITIAL_ELO)

matches_features = completed_matches.merge(
    home_features[
        [
            "date",
            "home_team_id",
            "home_rolling_scored",
            "home_rolling_conceded",
            "home_draw_rate_last_10",
        ]
    ],
    on=["date", "home_team_id"],
    how="left",
)

matches_features = matches_features.merge(
    away_features[
        [
            "date",
            "away_team_id",
            "away_rolling_scored",
            "away_rolling_conceded",
            "away_draw_rate_last_10",
        ]
    ],
    on=["date", "away_team_id"],
    how="left",
)

matches_features = matches_features.merge(
    elo_history,
    on="id",
    how="left",
)

matches_features["round_number"] = (
    matches_features["round"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
)
matches_features["round_number"] = pd.to_numeric(
    matches_features["round_number"],
    errors="coerce"
)

In [17]:
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["elo_diff"] = (
        df["home_elo"] - df["away_elo"]
    ).abs()

    df["attack_diff"] = (
        df["home_rolling_scored"] - df["away_rolling_scored"]
    ).abs()

    df["defense_diff"] = (
        df["home_rolling_conceded"] - df["away_rolling_conceded"]
    ).abs()

    df["combined_scoring"] = (
        df["home_rolling_scored"] + df["away_rolling_scored"]
    )

    df["combined_conceded"] = (
        df["home_rolling_conceded"] + df["away_rolling_conceded"]
    )

    df["total_balance"] = (
        df["attack_diff"] + df["defense_diff"]
    )

    df["points_diff_abs"] = df["points_diff"].abs()
    df["position_diff_abs"] = df["position_diff"].abs()

    df["draw_rate_diff"] = (
        df["home_draw_rate_last_10"] - df["away_draw_rate_last_10"]
    ).abs()

    df["draw_rate_sum"] = (
        df["home_draw_rate_last_10"] + df["away_draw_rate_last_10"]
    )

    df["home_advantage_strength"] = (
        df["home_rolling_scored"] - df["away_rolling_conceded"]
    )

    df["away_advantage_strength"] = (
        df["away_rolling_scored"] - df["home_rolling_conceded"]
    )

    df["strength_gap"] = (
        df["home_advantage_strength"] - df["away_advantage_strength"]
    ).abs()

    df["table_pressure_match"] = (
        df["home_title_race"] +
        df["away_title_race"] +
        df["home_europe_race"] +
        df["away_europe_race"] +
        df["home_relegation_risk"] +
        df["away_relegation_risk"]
    )

    df["low_scoring_profile"] = (
        df["combined_scoring"] <= 2.2
    ).astype(int)

    df["balanced_match_flag"] = (
        (df["elo_diff"] <= 50) &
        (df["position_diff_abs"] <= 3) &
        (df["points_diff_abs"] <= 6)
    ).astype(int)

    return df


linear_features = [
    "home_rolling_scored",
    "away_rolling_scored",
    "home_rolling_conceded",
    "away_rolling_conceded",
    "home_elo",
    "away_elo",
]

multiclass_features = [
    "home_elo",
    "away_elo",
    "home_rolling_scored",
    "away_rolling_scored",
    "home_rolling_conceded",
    "away_rolling_conceded",
    "home_draw_rate_last_10",
    "away_draw_rate_last_10",
    "home_position",
    "away_position",
    "points_diff",
    "position_diff",
    "home_title_race",
    "away_title_race",
    "home_europe_race",
    "away_europe_race",
    "home_relegation_risk",
    "away_relegation_risk",
    "elo_diff",
    "attack_diff",
    "defense_diff",
    "combined_scoring",
    "combined_conceded",
    "total_balance",
    "points_diff_abs",
    "position_diff_abs",
    "draw_rate_diff",
    "draw_rate_sum",
    "home_advantage_strength",
    "away_advantage_strength",
    "strength_gap",
    "table_pressure_match",
    "low_scoring_profile",
    "balanced_match_flag",
    "round_number",
]

matches_features = add_derived_features(matches_features)

base_feature_fill_cols = [
    "home_rolling_scored",
    "away_rolling_scored",
    "home_rolling_conceded",
    "away_rolling_conceded",
    "home_elo",
    "away_elo",
    "home_draw_rate_last_10",
    "away_draw_rate_last_10",
    "home_position",
    "away_position",
    "points_diff",
    "position_diff",
    "home_title_race",
    "away_title_race",
    "home_europe_race",
    "away_europe_race",
    "home_relegation_risk",
    "away_relegation_risk",
    "round_number",
]

for col in base_feature_fill_cols:
    matches_features[col] = pd.to_numeric(matches_features[col], errors="coerce")

base_feature_medians = matches_features[base_feature_fill_cols].median(numeric_only=True)

for col in base_feature_fill_cols:
    matches_features[col] = matches_features[col].fillna(base_feature_medians[col])

for col in linear_features + multiclass_features:
    matches_features[col] = pd.to_numeric(matches_features[col], errors="coerce")

In [18]:
linear_df = matches_features.dropna(subset=linear_features + ["home_dnb"]).copy()
multiclass_df = matches_features.dropna(subset=multiclass_features + ["match_result_3class"]).copy()

# Linear model: calibrated logistic regression without round
X_linear = linear_df[linear_features]
y_linear = linear_df["home_dnb"].astype(int)

linear_scaler = StandardScaler()
X_linear_scaled = linear_scaler.fit_transform(X_linear)

linear_base_model = LogisticRegression(max_iter=1000)
linear_model = CalibratedClassifierCV(
    linear_base_model,
    method="sigmoid",
    cv=5
)
linear_model.fit(X_linear_scaled, y_linear)

# Multiclass model
X_multi = multiclass_df[multiclass_features]
y_multi = multiclass_df["match_result_3class"].astype(int)

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=300,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="mlogloss",
)

xgb_model.fit(X_multi, y_multi)

feature_medians = base_feature_medians.to_dict()

In [19]:
teams_df = pd.read_sql(
    """
    SELECT
        id,
        name
    FROM teams
    """,
    conn
)

team_map = dict(zip(teams_df["id"], teams_df["name"]))


def get_future_matches(base_df: pd.DataFrame, competition_id: int, season: int | None = None) -> pd.DataFrame:
    mask = (
        (base_df["status"] == "NS") &
        (base_df["competition_id"] == competition_id)
    )

    if season is not None:
        mask &= (base_df["season"] == season)

    future = base_df.loc[mask].copy()

    if future.empty:
        return future

    future["competition_name"] = future["competition_id"].map(COMPETITION_ID_TO_NAME)
    future["home_team_name"] = future["home_team_id"].map(team_map)
    future["away_team_name"] = future["away_team_id"].map(team_map)

    return future


def predict_future_matches(base_df: pd.DataFrame, competition_id: int, season: int | None = None) -> pd.DataFrame:
    future = get_future_matches(base_df, competition_id, season)

    if future.empty:
        return future

    home_snapshot = team_snapshot.rename(
        columns={
            "team_id": "home_team_id",
            "rolling_goals_scored": "home_rolling_scored",
            "rolling_goals_conceded": "home_rolling_conceded",
            "draw_rate_last_10": "home_draw_rate_last_10",
            "elo": "home_elo",
        }
    )

    away_snapshot = team_snapshot.rename(
        columns={
            "team_id": "away_team_id",
            "rolling_goals_scored": "away_rolling_scored",
            "rolling_goals_conceded": "away_rolling_conceded",
            "draw_rate_last_10": "away_draw_rate_last_10",
            "elo": "away_elo",
        }
    )

    future = future.merge(
        home_snapshot[
            [
                "home_team_id",
                "home_rolling_scored",
                "home_rolling_conceded",
                "home_draw_rate_last_10",
                "home_elo",
            ]
        ],
        on="home_team_id",
        how="left",
    )

    future = future.merge(
        away_snapshot[
            [
                "away_team_id",
                "away_rolling_scored",
                "away_rolling_conceded",
                "away_draw_rate_last_10",
                "away_elo",
            ]
        ],
        on="away_team_id",
        how="left",
    )

    for col in base_feature_fill_cols:
        if col in future.columns:
            future[col] = pd.to_numeric(future[col], errors="coerce")
            future[col] = future[col].fillna(feature_medians[col])

    future = add_derived_features(future)

    future["round_number"] = (
        future["round"]
        .astype(str)
        .str.extract(r"(\d+)")[0]
    )
    
    future["round_number"] = pd.to_numeric(
        future["round_number"],
        errors="coerce"
    )
    
    for col in linear_features + multiclass_features:
        future[col] = pd.to_numeric(future[col], errors="coerce")
        future[col] = future[col].fillna(feature_medians.get(col, future[col].median()))

    linear_probs = linear_model.predict_proba(
        linear_scaler.transform(future[linear_features])
    )[:, 1]

    multi_probs = xgb_model.predict_proba(future[multiclass_features])

    future["linear_home_dnb_prob"] = linear_probs
    future["home_prob"] = multi_probs[:, 0]
    future["draw_prob"] = multi_probs[:, 1]
    future["away_prob"] = multi_probs[:, 2]

    future["top_diff"] = (
        future[["home_prob", "draw_prob", "away_prob"]]
        .apply(
            lambda row: np.sort(row.values)[-1] - np.sort(row.values)[-2],
            axis=1
        )
    )

    return future.sort_values(
        by=["competition_name", "top_diff", "date"],
        ascending=[True, True, True]
    ).reset_index(drop=True)


combined_forecast = predict_future_matches(
    matches_all,
    competition_id=FORECAST_COMPETITION_ID,
    season=FORECAST_SEASON
)

combined_forecast[
    [
        "date",
        "competition_name",
        "home_team_name",
        "away_team_name",
        "linear_home_dnb_prob",
        "home_prob",
        "draw_prob",
        "away_prob",
        "top_diff",
    ]
]

,date,competition_name,home_team_name,away_team_name,linear_home_dnb_prob,home_prob,draw_prob,away_prob,top_diff
0,2026-05-22 18:45:00,Serie A,Fiorentina,Atalanta,0.607087,0.433859,0.186632,0.379508,0.054351
1,2026-05-24 18:45:00,Serie A,Lecce,Genoa,0.644019,0.416493,0.254176,0.329331,0.087162
2,2026-05-23 16:00:00,Serie A,Bologna,Inter,0.380764,0.298971,0.213637,0.487392,0.188421
3,2026-05-24 13:00:00,Serie A,Parma,Sassuolo,0.693475,0.509150,0.225840,0.265010,0.244140
4,2026-05-24 18:45:00,Serie A,Hellas Verona,AS Roma,0.284198,0.194855,0.252031,0.553114,0.301082
5,2026-05-24 18:45:00,Serie A,Cremonese,Como,0.420155,0.255638,0.178511,0.565851,0.310212
6,2026-05-24 18:45:00,Serie A,Torino,Juventus,0.475863,0.228575,0.213289,0.558136,0.329560
7,2026-05-24 16:00:00,Serie A,Napoli,Udinese,0.860167,0.616611,0.217980,0.165409,0.398631
8,2026-05-24 18:45:00,Serie A,AC Milan,Cagliari,0.877250,0.632913,0.185746,0.181341,0.447167
9,2026-05-23 18:45:00,Serie A,Lazio,Pisa,0.910123,0.727453,0.155384,0.117162,0.572069
